# 04 · Model Evaluation
Confusion matrices, ROC/PR curves, feature importance, train vs test. Holdout is 2024-2025 (1128 county-months, unseen).


In [ ]:
from pathlib import Path
import json, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
ALT = Path("../../apps/backend/ml_artifacts")

# Feature cols with fallback
fpath = MODEL_DIR / "feature_columns.json"
if not fpath.exists():
    fpath = ALT / "feature_columns.json"
with open(fpath) as f:
    feature_cols = json.load(f)

# Test set with fallbacks
if (DATA_DIR / "X_test.csv").exists():
    X_test = pd.read_csv(DATA_DIR / "X_test.csv")[feature_cols]
    y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"].astype(int)
elif (DATA_DIR / "test_matrix.csv").exists():
    tm = pd.read_csv(DATA_DIR / "test_matrix.csv")
    X_test, y_test = tm[feature_cols], tm["target_risk"].astype(int)
else:
    base = pd.read_csv(DATA_DIR / "ml_base_2015_2025.csv")
    td = base[base["year"]>=2024]
    X_test, y_test = td[feature_cols], td["target_risk"].astype(int)

# Load models
models = {}
for name in ["Logistic Regression","Random Forest","XGBoost","LightGBM"]:
    p = MODEL_DIR / f"{name.lower().replace(' ','_')}.joblib"
    if p.exists():
        models[name] = joblib.load(p)
    else:
        print(f"Missing {p} — run 03 first")

print(f"Loaded {len(models)} models | Test {X_test.shape[0]} rows, {len(feature_cols)} feats | Shock {y_test.sum()} ({y_test.mean():.1%})")


### Confusion Matrices (False Negatives matter most for early warning)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,9))
axes = axes.flatten()
for ax, (name, m) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, m.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=ax, xticklabels=["Safe","Risk"], yticklabels=["Safe","Risk"])
    ax.set_title(f"{name}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {MODEL_DIR / 'confusion_matrices.png'}")


### ROC & PR Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
for name, m in models.items():
    prob = m.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax1.plot(fpr, tpr, label=f"{name} AUC={auc(fpr,tpr):.3f}")
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ax2.plot(rec, prec, label=f"{name} AP={average_precision_score(y_test, prob):.3f}")
ax1.plot([0,1],[0,1],"k--", label="Random")
ax1.set_title("ROC (2024-2025)"); ax1.set_xlabel("FPR"); ax1.set_ylabel("TPR (Recall)"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title("Precision-Recall"); ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / "roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()


### Feature Importance (tree models)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,6))
for ax, name in zip(axes, ["Random Forest","XGBoost","LightGBM"]):
    if name not in models:
        continue
    m = models[name]
    imp = getattr(m, "feature_importances_", None) or m.named_steps["clf"].feature_importances_ if hasattr(m, "named_steps") else getattr(m, "feature_importances_", None)
    s = pd.Series(imp, index=feature_cols).sort_values()
    ax.barh(s.index, s.values, color="#2e7d32")
    ax.set_title(f"{name}")
plt.tight_layout()
plt.savefig(MODEL_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


### Train vs Test


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
# Train with fallback
if (DATA_DIR / "X_train.csv").exists():
    X_train = pd.read_csv(DATA_DIR / "X_train.csv")[feature_cols]
    y_train = pd.read_csv(DATA_DIR / "y_train.csv")["target_risk"].astype(int)
else:
    tr = pd.read_csv(DATA_DIR / "train_matrix.csv")
    X_train, y_train = tr[feature_cols], tr["target_risk"].astype(int)

rows = []
for name, m in models.items():
    rows.append({
        "Model": name,
        "Train ROC-AUC": round(roc_auc_score(y_train, m.predict_proba(X_train)[:,1]),4),
        "Test ROC-AUC": round(roc_auc_score(y_test, m.predict_proba(X_test)[:,1]),4),
        "Test Acc": round(accuracy_score(y_test, m.predict(X_test)),4),
        "Test Recall": round(recall_score(y_test, m.predict(X_test)),4),
        "Test F1": round(f1_score(y_test, m.predict(X_test)),4),
    })
display(pd.DataFrame(rows).sort_values("Test F1", ascending=False))
